# ONNX Format | ONNX 格式

In this demo we are going to go through the process of exporting our best model to ONNX format and then using the runtime for inference.

在本演示中，我们将演示如何将最佳模型导出为 ONNX 格式，然后使用运行时进行推理。

Why do this?

为什么要这样做？

ONNX is an open standard format that enables model interoperability across different frameworks and platforms, making it easier to deploy models in diverse environments such as cloud, edge, or mobile devices. 

ONNX 是一种开放标准格式，支持跨不同框架和平台的模型互操作性，使得在云、边缘或移动设备等多样化环境中部署模型变得更加容易。

ONNX Runtime is highly optimized for performance, providing faster inference speeds through techniques like graph optimizations and support for hardware accelerators, including GPUs, CPUs, and specialized inference chips. This combination allows developers to achieve scalability, portability, and performance improvements, while simplifying integration into non-PyTorch ecosystems.

ONNX Runtime 在性能上进行了高度优化，通过图优化等技术以及对硬件加速器（包括 GPU、CPU 和专用推理芯片）的支持，提供更快的推理速度。这种组合使开发人员能够实现可扩展性、可移植性和性能改进，同时简化集成到非 PyTorch 生态系统中的过程。

In [ ]:
# Install the required modules | 安装所需的模块
%pip install onnx onnxruntime

In [ ]:
# RESTART YOUR NOTEBOOK FOR CHANGES TO TAKE | 重启笔记本以使更改生效

## Load our best model | 加载我们的最佳模型

Before we begin we must load our best model

在开始之前，我们必须加载最佳模型

In [ ]:
# Import modules | 导入模块
import torch
import torch.nn as nn
from torchvision import models

In [ ]:
# Load the mobilenet_v3_large model with default weights | 加载具有默认权重的 mobilenet_v3_large 模型
model = models.mobilenet_v3_large(weights=models.MobileNet_V3_Large_Weights.DEFAULT)

In [ ]:
# Modify last layer of the model for 2 classes as output | 修改模型的最后一层以输出 2 个类别
model.classifier[-1] = nn.Linear(1280, 2)

In [ ]:
# Load the model from checkpoint | 从检查点加载模型
checkpoint = torch.load('mobilenet_checkpoint.tar', weights_only=True)

In [ ]:
# Load the parameters from the checkpoint | 从检查点加载参数
model.load_state_dict(checkpoint['model_state_dict'])

## Export our Model to ONNX format | 将模型导出为 ONNX 格式

In [ ]:
# Import the module: NOTE that ONNX is built into PyTorch! | 导入模块：注意 ONNX 已内置于 PyTorch 中！
import torch.onnx

In [ ]:
# Read the helper function to export | 查看导出的辅助函数
help(torch.onnx.export)

In [ ]:
# Create an example output | 创建示例输出
example_input = torch.randn(1, 3, 224, 224)

In [ ]:
# Invoke export | 调用导出
torch.onnx.export(model, example_input, "image_classifier.onnx")

In [ ]:
# Check the model consistency | 检查模型一致性
import onnx

# Load it with ONNX | 使用 ONNX 加载
onnx_model = onnx.load("image_classifier.onnx")
# Check it | 检查
print(onnx.checker.check_model(onnx_model))

## Load an example image for inference | 加载示例图像进行推理

In [ ]:
# Transformations are still required | 仍然需要转换
from PIL import Image
from torchvision.transforms import v2

transform = v2.Compose([
    v2.Resize((224, 224)),
    v2.ToImage(), 
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.485, 0.456, 0.406], 
                 std=[0.229, 0.224, 0.225])
])

In [ ]:
# Open an image | 打开图像
image_path = 'sample-input.jpg'
image = Image.open(image_path)

In [ ]:
# Apply the transformation | 应用转换
transformed_image = transform(image)
transformed_image.shape

In [ ]:
# Add additional dimension due to requirements: [batch_size, channels, height, width] | 由于要求添加额外的维度：[batch_size, channels, height, width]
transformed_image = transformed_image.unsqueeze(0)
transformed_image.shape

In [ ]:
# Convert our transformed image to a Numpy Array | 将转换后的图像转换为 Numpy 数组
import numpy as np

image_np = np.array(transformed_image, dtype=np.float32)

## Run inference using ONNX Runtime | 使用 ONNX Runtime 运行推理

The ONNX Runtime is a high-performance inference engine designed to execute models in the open ONNX format across various platforms and devices. It optimizes model execution through graph-level optimizations and supports hardware accelerators, enabling fast, scalable, and portable deployments in diverse environments.

ONNX Runtime 是一个高性能推理引擎，旨在跨各种平台和设备执行开放的 ONNX 格式模型。它通过图级优化来优化模型执行，并支持硬件加速器，从而在多样化环境中实现快速、可扩展和可移植的部署。

In [ ]:
# Import the runtime | 导入运行时
import onnxruntime as ort

In [ ]:
# Load the model | 加载模型
import onnx

onnx_model = onnx.load("image_classifier.onnx")

In [ ]:
# Start on inference Session on the runtime | 在运行时启动推理会话
session = ort.InferenceSession("image_classifier.onnx")

In [ ]:
# Convert the image to a numpy array | 将图像转换为 numpy 数组
import numpy as np 

image_np = np.array(transformed_image, dtype=np.float32)

In [ ]:
# Run inference | 运行推理

# Create input to be passed to the model | 创建要传递给模型的输入
inputs = {session.get_inputs()[0].name: image_np}
# Run the inference | 运行推理
outputs = session.run(None, inputs)
print(outputs) # raw outputs (logits) from final layer | 最终层的原始输出（logits）

In [ ]:
# Get the predicted class | 获取预测的类别
predicted = outputs[0][0].argmax(0)
print(predicted)

In [ ]:
# Define our Dataset Class and label encoding | 定义数据集类和标签编码
label_encoding = {"malignant": 0, "benign": 1}

In [ ]:
# Reverse index the label_encoding dictionary | 反转 label_encoding 字典的索引
index_to_class_map = {v: k for k, v in label_encoding.items()}
print(f"Predicted Class: {index_to_class_map[predicted.item()]}")